In [1]:
from langchain_aws import ChatBedrockConverse
import os
from dotenv import load_dotenv

load_dotenv()
model = ChatBedrockConverse(
    model=os.getenv("AWS_MODEL_ID"),
    region_name=os.getenv("AWS_REGION"),)

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

/var/folders/sb/qd7jhvy55pdg4fjxlqjc6kz40000gn/T/ipykernel_23471/4208644247.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [3]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

Based on the search results, I can provide you with information about the number of artists in several prominent music databases:

## Music Databases and Their Artist Counts

1. **Encyclopaedia Metallum** - A heavy metal encyclopedia containing:
   - **200,184 bands**
   - **974,613 artists**

2. **Viberate** - Music discovery platform with:
   - **Over 1 million artists** in their database
   - Lists Taylor Swift, Shakira, Justin Bieber, Ariana Grande, Bad Bunny, and Drake among their top artists

3. **AllMusic (AMG)** - Music information and reviews database
   - Contains artist information though the exact count isn't specified in the results

4. **MusicBrainz** - Community-maintained music metadata database
   - Contains comprehensive information about artists, but the exact count isn't provided in the results

The most specific numbers come from Encyclopaedia Metallum, which shows nearly 1 million artists (including both bands and individual artists), while Viberate explicitly men